# Primary Model Evaluation (Mean + FPR@20%)

This notebook is the **primary evaluation** for all trained autoencoders using:
- **Score method:** mean reconstruction error
- **Thresholding:** FPR-controlled threshold on normal validation scores
- **Target FPR:** 20%

This avoids label-optimized threshold bias (e.g., Youden on mixed eval labels) and keeps one consistent policy across models.

## Evaluation order
1. ECNN-AE Optimized
2. CNN-AE Large
3. CNN-AE Augmented
4. CNN-AE
5. ResNet-AE
6. ResNet-AE Finetuned (partial)

In [ ]:
# Colab / local environment setup
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path('/content/symAD-ECNN')
    if not REPO_ROOT.exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/RifaDeen/symAD-ECNN.git', str(REPO_ROOT)])
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent:
        if (REPO_ROOT / 'README.md').exists() and (REPO_ROOT / 'notebooks').exists():
            break
        REPO_ROOT = REPO_ROOT.parent

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
for p in [REPO_ROOT, EVALS_DIR, EVALS_DIR / 'ecnn_thresholding']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f'IN_COLAB: {IN_COLAB}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'EVALS_DIR: {EVALS_DIR}')

In [ ]:
# Imports
import json
import shutil
import zipfile
from glob import glob
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import confusion_matrix, roc_curve, auc

from metrics_utils import threshold_from_normal_scores, compute_full_metrics

try:
    from ecnn_model_loader import get_model_for_inference
except Exception:
    # Required for ECNN model loading
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'e2cnn', '-q'])
    from ecnn_model_loader import get_model_for_inference

plt.style.use('seaborn-v0_8')
print('All imports ready.')

In [ ]:
# Paths, output folders, and model configuration
if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
    OUTPUT_ROOT = PROJECT_ROOT / 'evaluations'
else:
    PROJECT_ROOT = REPO_ROOT
    OUTPUT_ROOT = REPO_ROOT / 'results' / 'primary_evaluations'

DATA_ROOT = PROJECT_ROOT / 'data'
MODELS_ROOT = PROJECT_ROOT / 'models' / 'saved_models'

PRIMARY_NAME = 'primary_mean_fpr20'
TABLE_DIR = OUTPUT_ROOT / 'tables' / PRIMARY_NAME
FIG_DIR = OUTPUT_ROOT / 'figures' / PRIMARY_NAME
JSON_DIR = OUTPUT_ROOT / 'json' / PRIMARY_NAME
for d in [TABLE_DIR, FIG_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_SCORE_METHOD = 'mean'
TARGET_FPR = 0.20
CLASS_NAMES = ['Normal', 'Anomaly']

# Match ECNN training notebook turbo loader paths
BASE_DIR = DATA_ROOT
LOCAL_DIR = Path('/content/local_data') if IN_COLAB else (REPO_ROOT / '.local_data')

ZIPS = {
    'train': BASE_DIR / 'train_fast.zip',
    'val':   BASE_DIR / 'val_fast.zip',
    'test':  BASE_DIR / 'test_fast.zip',
}

IXI_TRAIN_PATH = LOCAL_DIR / 'train'
IXI_VAL_PATH = LOCAL_DIR / 'val'
BRATS_PATH = LOCAL_DIR / 'test'

MODEL_CONFIGS = [
    {
        'key': 'ecnn_opt',
        'display_name': 'ECNN-AE (Optimized)',
        'model_type': 'ecnn',
        'checkpoint_dirs': ['ecnn_optimized', '.'],
        'checkpoint_patterns': ['ecnn_optimized_best.pth', 'ecnn_best.pth', '*ecnn*optimized*best*.pth', '*ecnn*best*.pth'],
    },
    {
        'key': 'cnn_large',
        'display_name': 'CNN-AE Large',
        'model_type': 'cnn_large',
        'checkpoint_dirs': ['cnn_ae_large'],
        'checkpoint_patterns': ['cnn_large_best.pth', '*cnn*large*best*.pth'],
    },
    {
        'key': 'cnn_aug',
        'display_name': 'CNN-AE Augmented',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae_augmented'],
        'checkpoint_patterns': ['cnn_aug_best.pth', '*cnn*aug*best*.pth'],
    },
    {
        'key': 'cnn_base',
        'display_name': 'CNN-AE',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae'],
        'checkpoint_patterns': ['cnn_best.pth', '*cnn*best*.pth'],
    },
    {
        'key': 'resnet_ae',
        'display_name': 'ResNet-AE',
        'model_type': 'resnet_frozen',
        'checkpoint_dirs': ['resnet_ae'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
    {
        'key': 'resnet_ft',
        'display_name': 'ResNet-AE Finetuned (partial)',
        'model_type': 'resnet_finetuned_partial',
        'checkpoint_dirs': ['resnet_finetuned_partial', 'resnet_finetuned_full', 'resnet_finetuned_none'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
]

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'MODELS_ROOT: {MODELS_ROOT}')
print(f'TABLE_DIR: {TABLE_DIR}')
print(f'FIG_DIR: {FIG_DIR}')
print(f'JSON_DIR: {JSON_DIR}')

In [ ]:
# Shared model classes + helper functions
from model_defs import CNNAutoencoder, LargeCNNAutoencoder, ResNetAutoencoder
from eval_common import (
    find_files,
    extract_zip,
    resolve_checkpoint_path as resolve_checkpoint_path_common,
    get_state_dict as get_state_dict_common,
    compute_reconstruction_errors,
)

class SliceDataset(Dataset):
    def __init__(self, files: List[Path], mode: str = 'grayscale'):
        self.files = files
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.files)

    def _load_slice(self, path: Path) -> np.ndarray:
        if path.suffix.lower() == '.npy':
            arr = np.load(path)
        else:
            arr = np.array(Image.open(path).convert('L')).astype(np.float32) / 255.0
        arr = arr.astype(np.float32)
        if arr.max() > 1.0:
            arr = arr / 255.0
        if arr.ndim == 3:
            arr = arr.squeeze()
        return arr

    def __getitem__(self, idx):
        arr = self._load_slice(self.files[idx])
        gray = torch.from_numpy(arr).float().unsqueeze(0)
        if gray.shape[-2:] != (128, 128):
            gray = F.interpolate(gray.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)

        if self.mode == 'resnet':
            img_uint8 = (gray.squeeze(0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            tgt = gray
            return inp, tgt

        return gray, gray
def check_turbo_zips(zips: Dict[str, Path]):
    missing = [str(p) for p in zips.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            'Missing turbo zip files. Run training/turbo prep first. Missing: ' + ', '.join(missing)
        )


def resolve_checkpoint_path(cfg: Dict) -> Path:
    return resolve_checkpoint_path_common(cfg, MODELS_ROOT)


def load_model_for_config(cfg: Dict, device: str):
    ckpt_path = resolve_checkpoint_path(cfg)

    if cfg['model_type'] == 'ecnn':
        model, _ = get_model_for_inference(ckpt_path, device)
        model.eval()
        return model, ckpt_path

    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = get_state_dict_common(checkpoint)

    if cfg['model_type'] == 'cnn_large':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 512
        model = LargeCNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'cnn_base':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 256
        model = CNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'resnet_frozen':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['model_type'] == 'resnet_finetuned_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    else:
        raise ValueError(f"Unsupported model type: {cfg['model_type']}")

    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model, ckpt_path


def compute_errors(model: nn.Module, dataloader: DataLoader, device: str) -> np.ndarray:
    return compute_reconstruction_errors(model, dataloader, device)


print('Shared model imports + helper definitions ready.')

In [ ]:
# Run primary evaluation for all models (same turbo loader flow as training notebook)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

check_turbo_zips(ZIPS)

print('Extracting turbo datasets to local disk...')
for split, zip_path in ZIPS.items():
    target_dir = LOCAL_DIR / split
    extract_zip(zip_path, target_dir)
    print(f'  {split}: {zip_path} -> {target_dir}')

train_files = find_files(IXI_TRAIN_PATH)
normal_files = find_files(IXI_VAL_PATH)
anomaly_files = find_files(BRATS_PATH)

print(f'Train files (IXI train): {len(train_files)}')
print(f'Normal files (IXI val): {len(normal_files)}')
print(f'Anomaly files (BraTS test): {len(anomaly_files)}')
if len(normal_files) == 0 or len(anomaly_files) == 0:
    raise RuntimeError('No data files found after extraction. Check zip contents.')

rows = []
roc_payload = []

for cfg in MODEL_CONFIGS:
    print(f"\n=== Evaluating: {cfg['display_name']} ===")

    model, ckpt_path = load_model_for_config(cfg, device)
    mode = 'resnet' if 'resnet' in cfg['model_type'] else 'grayscale'

    normal_ds = SliceDataset(normal_files, mode=mode)
    anomaly_ds = SliceDataset(anomaly_files, mode=mode)

    normal_loader = DataLoader(normal_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    anomaly_loader = DataLoader(anomaly_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    normal_scores = compute_errors(model, normal_loader, device)
    anomaly_scores = compute_errors(model, anomaly_loader, device)

    threshold = threshold_from_normal_scores(normal_scores, target_fpr=TARGET_FPR)

    y_true = np.concatenate([np.zeros(len(normal_scores), dtype=int), np.ones(len(anomaly_scores), dtype=int)])
    y_scores = np.concatenate([normal_scores, anomaly_scores])

    metrics = compute_full_metrics(y_true, y_scores, threshold)
    y_pred = (y_scores >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    # Confusion Matrix Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('True', fontsize=12, fontweight='bold')
    axes[0].set_title(f"{cfg['display_name']} - Confusion Matrix", fontsize=13, fontweight='bold')

    cm_norm = cm.astype('float') / np.clip(cm.sum(axis=1)[:, np.newaxis], a_min=1, a_max=None)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
                xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[1].set_xlabel('Predicted', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('True', fontsize=12, fontweight='bold')
    axes[1].set_title('Normalized', fontsize=13, fontweight='bold')

    plt.tight_layout()
    cm_path = FIG_DIR / f"{cfg['key']}_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f'Confusion matrix saved: {cm_path}')

    fpr_vals, tpr_vals, _ = roc_curve(y_true, y_scores)
    roc_payload.append({
        'model_name': cfg['display_name'],
        'fpr': fpr_vals,
        'tpr': tpr_vals,
        'auroc': float(metrics['auroc']),
    })

    row = {
        'model_key': cfg['key'],
        'model_name': cfg['display_name'],
        'checkpoint_path': str(ckpt_path),
        'score_method': TARGET_SCORE_METHOD,
        'threshold_method': 'fpr',
        'target_fpr': TARGET_FPR,
        'threshold': float(threshold),
        'accuracy': float(metrics['accuracy']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'specificity': float(metrics['specificity']),
        'f1_score': float(metrics['f1_score']),
        'auroc': float(metrics['auroc']),
        'auprc': float(metrics['auprc']),
        'tp': int(metrics['tp']),
        'tn': int(metrics['tn']),
        'fp': int(metrics['fp']),
        'fn': int(metrics['fn']),
        'fpr': float(metrics['fpr']),
        'fnr': float(metrics['fnr']),
        'total_samples': int(metrics['total_samples']),
        'total_positive': int(metrics['total_positive']),
        'total_negative': int(metrics['total_negative']),
        'normal_count': int(len(normal_scores)),
        'anomaly_count': int(len(anomaly_scores)),
        'normal_mean': float(np.mean(normal_scores)),
        'normal_std': float(np.std(normal_scores)),
        'anomaly_mean': float(np.mean(anomaly_scores)),
        'anomaly_std': float(np.std(anomaly_scores)),
    }
    rows.append(row)

    np.save(JSON_DIR / f"{cfg['key']}_normal_scores.npy", normal_scores)
    np.save(JSON_DIR / f"{cfg['key']}_anomaly_scores.npy", anomaly_scores)

print('\nPrimary evaluation run complete.')

In [ ]:
# Save consolidated tables and rankings
if not rows:
    raise RuntimeError('No model results collected.')

df = pd.DataFrame(rows).sort_values('auroc', ascending=False).reset_index(drop=True)

display_cols = [
    'model_name', 'score_method', 'threshold_method', 'target_fpr', 'threshold',
    'accuracy', 'precision', 'recall', 'specificity', 'f1_score', 'auroc', 'auprc',
    'tp', 'tn', 'fp', 'fn', 'total_samples', 'total_positive', 'total_negative'
]
display(df[display_cols])

csv_path = TABLE_DIR / 'primary_mean_fpr20_metrics.csv'
md_path = TABLE_DIR / 'primary_mean_fpr20_metrics.md'
json_path = JSON_DIR / 'primary_mean_fpr20_metrics.json'

df.to_csv(csv_path, index=False)
df.to_markdown(md_path, index=False)
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(rows, f, indent=2)

best_path = TABLE_DIR / 'primary_mean_fpr20_best_model.txt'
best = df.iloc[0]
with open(best_path, 'w', encoding='utf-8') as f:
    f.write('Primary Evaluation (mean + fpr@0.20)\n')
    f.write(f"Best model by AUROC: {best['model_name']}\n")
    f.write(f"AUROC={best['auroc']:.4f}, AUPRC={best['auprc']:.4f}, F1={best['f1_score']:.4f}, Recall={best['recall']:.4f}, Specificity={best['specificity']:.4f}\n")

print(f'CSV saved: {csv_path}')
print(f'Markdown saved: {md_path}')
print(f'JSON saved: {json_path}')
print(f'Best-model note saved: {best_path}')

In [ ]:
# Combined ROC plot for all models
plt.figure(figsize=(9, 7))
for item in roc_payload:
    plt.plot(item['fpr'], item['tpr'], linewidth=2, label=f"{item['model_name']} (AUROC={item['auroc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Primary ROC Comparison (mean + fpr@0.20)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
roc_path = FIG_DIR / 'primary_mean_fpr20_roc_comparison.png'
plt.savefig(roc_path, dpi=150)
plt.show()
print(f'Combined ROC saved: {roc_path}')

## Done
This notebook is now your organized **primary evaluation** pipeline with a fixed and consistent thresholding policy (**mean + FPR 20%**), full classification metrics, totals, and saved confusion matrix visualizations for each model.